# Scenario 04 — Source Inactivation

Publishing new tasks against an inactive source reverts. This notebook shows the
full lifecycle: active → inactive → the revert that follows.

**Role:** Client only.

**Flow:**

```
Client
  │
  ├─ 1. publish_source         → ACTIVE
  ├─ 2. source.inactivate      → Receipt
  ├─ 3. get_status()            → INACTIVE
  └─ 4. publish_task (expected to revert)
```


## 0. Setup


In [ ]:
import os
import time

from web3 import Web3

from ogpu import ChainConfig, ChainId
from ogpu.client import (
    DeliveryMethod,
    ImageEnvironments,
    SourceInfo,
    TaskInfo,
    TaskInput,
    publish_source,
    publish_task,
)
from ogpu.protocol import Master, Provider, Response, Source, Task, vault

ChainConfig.set_chain(ChainId.OGPU_TESTNET)

CLIENT_KEY = os.environ["CLIENT_PRIVATE_KEY"]


## 1. Publish a source


In [ ]:
info = SourceInfo(
    name="inactivate-demo",
    description="scenario 04",
    logoUrl="https://example.com/logo.png",
    imageEnvs=ImageEnvironments(
        cpu="https://cipfs.ogpuscan.io/ipfs/QmNWFLL13ujf3KUTJvfNx42bA5fWDV96qqUdjY6nwpuwD9",
    ),
    minPayment=Web3.to_wei(0.01, "ether"),
    minAvailableLockup=0,
    maxExpiryDuration=86400,
)
source = publish_source(info, private_key=CLIENT_KEY)
print(f"Source: {source.address}  status={source.get_status()}")


## 2. Inactivate


In [ ]:
receipt = source.inactivate(signer=CLIENT_KEY)
print(f"Inactivate tx: {receipt.tx_hash}")


## 3. Verify


In [ ]:
from ogpu.types import SourceStatus

s = source.get_status()
print(f"Status: {s}")
assert s == SourceStatus.INACTIVE


## 4. Attempt to publish against the inactive source

This should revert with a typed `SourceInactiveError` (or a generic `TxRevertError` if the
revert string is not in `REVERT_PATTERN_MAP`).


In [ ]:
from ogpu.types import OGPUError

try:
    publish_task(
        TaskInfo(
            source=source.address,
            config=TaskInput(function_name="x", data={}),
            expiryTime=int(time.time()) + 3600,
            payment=Web3.to_wei(0.01, "ether"),
        ),
        private_key=CLIENT_KEY,
    )
except OGPUError as e:
    print(f"Reverted as expected: {type(e).__name__}: {e}")
